# 20 LSTM_SEEDS: seed variability of the LSTM forecaster

Repeats the LSTM of `14_FORECAST` with five seeds fixed in advance (42, 1, 2, 3, 4), as `08_SEEDS` does for the classifiers, so that the spread due to the seed can be set against the difference between the LSTM and SARIMA.

* **What is repeated.** The LSTM of `14_FORECAST`, unchanged: the same daily series (out-of-service days masked), 30-day blind horizon, lookback of 14 days, architecture (one LSTM layer of 50 units and a dense output), optimiser, batch size, early stopping and recursive forecast. Only the seed changes.
* **Seeding.** Under Keras 3, weight initialisers without an explicit seed draw from Python's `random` module, so seeding TensorFlow and NumPy alone, as in `14_FORECAST`, does not fix the initial weights. Here `keras.utils.set_random_seed` seeds Python, NumPy and TensorFlow together and op determinism is switched on, so each seed gives the same result on every run. The single LSTM result of `14_FORECAST` is printed alongside for information only.
* **What is not repeated.** SARIMA and the naive baselines have no random component; their values are read from `forecast_metrics.csv` (written by `14_FORECAST`), and the baselines are recomputed as a check.
* **Environment.** Python 3.11 with TensorFlow, as `14_FORECAST`. Each seed is written to the results file as it finishes; seeds already there are skipped on a rerun.

Outputs (`OUTPUT/v4/`): `rev_lstm_seeds.csv` (one row per seed), `rev_lstm_seeds_forecasts.csv` (the 30 daily forecasts of each seed), `rev_lstm_seeds_summary.csv` (mean, SD, minimum and maximum of each measure, with SARIMA and the baselines).

Paper: Sections 2.7.4 and 3.3; Table 11.

In [ ]:
import os, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from sklearn.preprocessing import MinMaxScaler
import sys, tensorflow as tf, keras
tf.config.experimental.enable_op_determinism()
print(f"Python {sys.version.split()[0]}, TensorFlow {tf.__version__}, Keras {keras.__version__}")

CFG = dict(
    episodes="build/fault_episodes_v4.csv",
    metrics="OUTPUT/v4/forecast_metrics.csv",
    outdir="OUTPUT/v4",
    horizon=30,
    m=7,
    lookback=14,
    gap_min_days=7,
    seeds=[42, 1, 2, 3, 4],
)
OUT_RUNS = f"{CFG['outdir']}/rev_lstm_seeds.csv"
OUT_FC = f"{CFG['outdir']}/rev_lstm_seeds_forecasts.csv"
OUT_SUM = f"{CFG['outdir']}/rev_lstm_seeds_summary.csv"

## 1. Daily series, baselines and metrics (as in `14_FORECAST`)

In [ ]:
def daily_counts(cfg):
    ep = pd.read_csv(cfg["episodes"], parse_dates=["onset"])
    s = (ep.set_index("onset").resample("D").size()
           .asfreq("D").fillna(0).astype(float))
    s.name = "faults"
    zero = s.eq(0)
    block = (zero != zero.shift()).cumsum()
    for _, idx in s.groupby(block).groups.items():
        idx = pd.DatetimeIndex(idx)
        if zero.loc[idx].all() and len(idx) >= cfg["gap_min_days"]:
            s.loc[idx] = np.nan
            print(f"masked {idx.min().date()} to {idx.max().date()} ({len(idx)} days)")
    return s


def naive(train, h):
    return np.repeat(train.dropna().iloc[-1], h)


def seasonal_naive(train, h, m):
    last = train.ffill().iloc[-m:].values
    return np.array([last[i % m] for i in range(h)])


def score(y, pred, train, m):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    mae = np.mean(np.abs(y - pred))
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    d = train.values[m:] - train.values[:-m]
    scale = np.nanmean(np.abs(d))
    return dict(MAE=mae, RMSE=rmse, MASE=mae / scale if scale else np.nan)


s = daily_counts(CFG)
h = CFG["horizon"]
train, test = s.iloc[:-h], s.iloc[-h:]
print(f"{len(s)} days; train {len(train)} ({train.notna().sum()} observed); "
      f"test {test.index.min().date()} to {test.index.max().date()}")

ref = pd.read_csv(CFG["metrics"], index_col=0)
chk = {"Naive": score(test, naive(train, h), train, CFG["m"]),
       f"Seasonal naive (m={CFG['m']})": score(test, seasonal_naive(train, h, CFG["m"]), train, CFG["m"])}
for k, v in chk.items():
    same = abs(round(v["MASE"], 3) - ref.loc[k, "MASE"]) < 1e-9
    print(f"check {k:22s} MASE {v['MASE']:.3f}  stored {ref.loc[k, 'MASE']:.3f}  "
          f"{'OK' if same else 'DIFFERS: check inputs'}")

## 2. LSTM with a given seed (`fit_lstm` of `14_FORECAST`, seeded as above, epochs recorded)

In [ ]:
def fit_lstm(train, h, cfg, seed):
    import tensorflow as tf, keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Input

    L = cfg["lookback"]
    sc = MinMaxScaler().fit(train.dropna().values.reshape(-1, 1))
    z = sc.transform(train.values.reshape(-1, 1)).ravel()
    keep = [i for i in range(len(z) - L) if not np.isnan(z[i:i + L + 1]).any()]
    X = np.array([z[i:i + L] for i in keep])[..., None]
    y = np.array([z[i + L] for i in keep])
    cut = int(len(X) * 0.85)
    Xtr, ytr, Xva, yva = X[:cut], y[:cut], X[cut:], y[cut:]

    tf.keras.backend.clear_session()
    keras.utils.set_random_seed(seed)      # Python, NumPy and TensorFlow together
    model = Sequential([Input((L, 1)), LSTM(50), Dense(1)])
    model.compile(optimizer="adam", loss="mse")
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=200, batch_size=16,
                     verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(
                         patience=20, restore_best_weights=True)])

    tail = train.ffill().bfill().values[-L:].reshape(-1, 1)
    window = list(sc.transform(tail).ravel())
    preds = []
    for _ in range(h):
        p = float(model.predict(np.array(window[-L:])[None, :, None], verbose=0)[0, 0])
        preds.append(p)
        window.append(p)
    pred = sc.inverse_transform(np.array(preds).reshape(-1, 1)).ravel()
    vl = hist.history["val_loss"]
    return pred, dict(sequences=len(X), epochs_run=len(vl),
                      best_epoch=int(np.argmin(vl)) + 1, best_val_loss=float(np.min(vl)))

## 3. Run the five seeds (resumable)

In [ ]:
done = pd.read_csv(OUT_RUNS) if os.path.exists(OUT_RUNS) else pd.DataFrame()
fcs = pd.read_csv(OUT_FC, index_col=0, parse_dates=True) if os.path.exists(OUT_FC) \
      else pd.DataFrame(index=test.index)

for seed in CFG["seeds"]:
    if len(done) and seed in set(done["seed"]):
        print(f"seed {seed}: already done, skipped")
        continue
    t0 = time.time()
    pred, info = fit_lstm(train, h, CFG, seed)
    row = dict(seed=seed, **score(test, pred, train, CFG["m"]), **info,
               seconds=round(time.time() - t0, 1))
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
    done.to_csv(OUT_RUNS, index=False)
    fcs[f"seed_{seed}"] = pred
    fcs.assign(actual=test.values).to_csv(OUT_FC)
    print(f"seed {seed}: MAE {row['MAE']:.3f}  RMSE {row['RMSE']:.3f}  MASE {row['MASE']:.3f}  "
          f"(best epoch {info['best_epoch']} of {info['epochs_run']}, {row['seconds']} s)")

done = done.set_index("seed").loc[CFG["seeds"]]
print()
print(done.round(3).to_string())

## 4. Summary and comparison with SARIMA

In [ ]:
m = ["MAE", "RMSE", "MASE"]
summ = pd.DataFrame({
    "LSTM mean": done[m].mean(),
    "LSTM SD": done[m].std(ddof=1),
    "LSTM min": done[m].min(),
    "LSTM max": done[m].max(),
}).T
for k in ["SARIMA", f"Seasonal naive (m={CFG['m']})", "Naive"]:
    summ.loc[k] = ref.loc[k, m]
summ = summ.round(3)
summ.to_csv(OUT_SUM)
print(summ.to_string())

r42 = done.loc[42]
print(f"\n14_FORECAST LSTM MASE {ref.loc['LSTM', 'MASE']:.3f} (single run, initial weights not seeded); "
      f"seed 42 here {r42['MASE']:.3f}")

sar = ref.loc["SARIMA"]
print("\nLSTM against SARIMA, by metric:")
for k in m:
    lo, hi = done[k].min(), done[k].max()
    beat = int((done[k] < sar[k]).sum())
    where = "inside" if lo <= sar[k] <= hi else ("below" if sar[k] < lo else "above")
    print(f"  {k:4s} SARIMA {sar[k]:.3f}; LSTM {done[k].mean():.3f} \u00b1 {done[k].std(ddof=1):.3f} "
          f"(range {lo:.3f} to {hi:.3f}); SARIMA {where} the LSTM range; "
          f"LSTM lower in {beat} of {len(done)} seeds")
print("\nIf SARIMA lies inside the LSTM range on MASE, the two forecasters are reported as comparable.")

___
**END**